# Guardrail Evaluation Report
This notebook only loads precomputed artifacts and renders analysis visualizations.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
required = [
    "artifacts/metrics/llm_before_after_summary.json",
    "artifacts/metrics/llm_before_after_runtime.json",
    "artifacts/predictions/llm_before_after.jsonl",
]
missing = [p for p in required if not (ROOT / p).exists()]
if missing:
    raise FileNotFoundError("Missing required artifact files: " + ", ".join(missing))
print("Notebook preconditions satisfied.")

In [ ]:
summary = json.loads(
    (ROOT / "artifacts/metrics/llm_before_after_summary.json").read_text()
)
runtime = json.loads(
    (ROOT / "artifacts/metrics/llm_before_after_runtime.json").read_text()
)
results = pd.read_json(
    ROOT / "artifacts/predictions/llm_before_after.jsonl", lines=True
)

scenario_summary = pd.DataFrame(
    [
        {
            "scenario": "before_no_guardrail",
            "attack_samples": summary["counts"]["attack_rows"],
            "successful_injections": summary["before"]["success_count"],
            "injection_success_rate": summary["before"]["success_rate"],
            "benign_refusal_rate": summary["before"]["benign_refusal_rate"],
        },
        {
            "scenario": "after_with_guardrail",
            "attack_samples": summary["counts"]["attack_rows"],
            "successful_injections": summary["after"]["success_count"],
            "injection_success_rate": summary["after"]["success_rate"],
            "benign_refusal_rate": summary["after"]["benign_refusal_rate"],
        },
    ]
)

print("Run metadata:", summary["run_metadata"])
print("Runtime summary:", runtime)
scenario_summary

In [ ]:
before_success_instances = results[
    (results["label"] == 1) & (results["before_injection_success"] == True)
][["id", "category", "text", "before_response", "source"]].reset_index(drop=True)

after_success_instances = results[
    (results["label"] == 1) & (results["after_injection_success"] == True)
][
    ["id", "category", "text", "after_response", "guardrail_reason", "source"]
].reset_index(
    drop=True
)

before_success_instances["before_response_excerpt"] = (
    before_success_instances["before_response"].astype(str).str.slice(0, 220)
)
after_success_instances["after_response_excerpt"] = (
    after_success_instances["after_response"].astype(str).str.slice(0, 220)
)

print("Before successful injection count:", len(before_success_instances))
print("After successful injection count:", len(after_success_instances))

## Notebook Scope
This notebook does not run model inference or guardrail evaluation logic.
It only loads source-generated before/after artifacts and shows:
- Specific successful prompt injection instances
- Before/after results tables
- Plots of effectiveness and latency

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(scenario_summary["scenario"], scenario_summary["injection_success_rate"])
ax.set_ylim(0, 1)
ax.set_title("Prompt Injection Success Rate: Before vs After")
ax.set_ylabel("Success Rate")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()

In [ ]:
category_rows = []
for category, payload in summary["per_category"].items():
    category_rows.append(
        {
            "category": category,
            "before_success_rate": payload["before_success_rate"],
            "after_success_rate": payload["after_success_rate"],
        }
    )

cat_df = pd.DataFrame(category_rows).sort_values("before_success_rate", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(cat_df))
ax.bar(
    [i - 0.2 for i in x],
    cat_df["before_success_rate"],
    width=0.4,
    label="before_no_guardrail",
)
ax.bar(
    [i + 0.2 for i in x],
    cat_df["after_success_rate"],
    width=0.4,
    label="after_with_guardrail",
)
ax.set_xticks(list(x), cat_df["category"], rotation=35, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Injection Success Rate")
ax.set_title("Per-Category Injection Success Rate")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
print("Successful prompt injections before guardrail (actual instances):")
if len(before_success_instances) == 0:
    print("None observed.")
else:
    display(
        before_success_instances[
            ["id", "category", "text", "before_response_excerpt", "source"]
        ]
    )

In [ ]:
print("Successful prompt injections after guardrail (residual misses):")
if len(after_success_instances) == 0:
    print("None observed in this run.")
else:
    display(
        after_success_instances[
            [
                "id",
                "category",
                "text",
                "after_response_excerpt",
                "guardrail_reason",
                "source",
            ]
        ]
    )

In [ ]:
latency_df = results[
    ["before_latency_ms", "after_latency_ms", "guardrail_latency_ms"]
].melt(var_name="stage", value_name="latency_ms")

fig, ax = plt.subplots(figsize=(8, 4))
for stage_name, group in latency_df.groupby("stage"):
    ax.hist(group["latency_ms"], bins=20, alpha=0.5, label=stage_name)
ax.set_title("Latency Distribution by Stage")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.legend()
fig.tight_layout()
plt.show()

interactive_latency = px.box(
    latency_df,
    x="stage",
    y="latency_ms",
    points="all",
    title="Latency by Stage (Interactive)",
)
interactive_latency.show()

## Specific Prompt Injection Instances and Result Slices
The tables below show exact prompt texts that successfully injected the model before guardrails,
and any residual successful injections after guardrails.

In [ ]:
top_before = before_success_instances[
    ["id", "category", "text", "before_response_excerpt", "source"]
].head(10)
top_after = after_success_instances[
    ["id", "category", "text", "after_response_excerpt", "guardrail_reason", "source"]
].head(10)

print("Top successful injections before guardrail:")
if len(top_before) == 0:
    print("None observed.")
else:
    display(top_before)

print("\nTop successful injections after guardrail (if any):")
if len(top_after) == 0:
    print("None observed in this run.")
else:
    display(top_after)

reduction = summary["delta"]["success_rate_reduction"]
print(f"\nBefore-to-after success-rate reduction: {reduction:.3f}")